### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [112]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [113]:
def process_all_pdfs(pdf_dir):
    """Process all PDF files in the specified directory and return a list of documents with metadata."""
    all_documents = []
    pdf_dir_path = Path(pdf_dir)

    pdf_files = list(pdf_dir_path.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source metadata to each document, additional metadata can be added here as needed
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"\nLoaded {len(documents)} documents from {pdf_file}.")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_docs = process_all_pdfs("../data/pdf")
print(f"\nTotal documents processed: {len(all_docs)}")

Found 2 PDF files to process.

Processing ../data/pdf/attentionisallyouneed.pdf...

Loaded 15 documents from ../data/pdf/attentionisallyouneed.pdf.

Processing ../data/pdf/rag-deck.pdf...

Loaded 20 documents from ../data/pdf/rag-deck.pdf.

Total documents loaded: 35

Total documents processed: 35


In [114]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter for RAG processing."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nSample split document metadata: {split_docs[0].metadata}")
        print(f"\nSample split document content (first 200 chars): {split_docs[0].page_content[:200]}")
    return split_docs

In [115]:
chunked_docs = split_documents(all_docs)
print(f"\nTotal chunked documents: {len(chunked_docs)}")

Split 35 documents into 72 chunks.

Sample split document metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/attentionisallyouneed.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attentionisallyouneed.pdf', 'file_type': 'pdf'}

Sample split document content (first 200 chars): Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need


Total chunked documents: 72


### Embedding and VectorstoreDB

In [116]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [117]:
class EmbeddingManager:
    """Manages embedding generation using SentenceTransformer and storage/retrieval in ChromaDB."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.
        
        Args:
            model_name (str): The name of the HuggingFace model to use for embeddings.
            """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        Returns:
            np.ndarray: An array of embeddings corresponding to the input texts.   
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded.") 
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# Initialize the embedding manager and generate embeddings for the chunked documents
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6424.56it/s]


Model loaded successfully. Embedding dimension: 384


## VectorStore 

In [118]:
class VectorStore:
    """Manages vector storage and retrieval using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store.
        
        Args:
            collection_name (str): The name of the ChromaDB collection to use for storing vectors.
            persist_directory (str): The directory where ChromaDB will persist its data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"descrption": "Collection for storing PDF document embeddings for RAG pipelines"}
            )
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully at '{self.persist_directory}'.")
            print(f"Existing documents in collection: {len(self.collection.get(ids=None))}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """Add documents to the vector store with generated embeddings.
        
        Args:
            documents: List of documents to add, where each document is a dict with 'page_content' and 'metadata'.
            embeddings: List of embeddings corresponding to the documents.
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")
        
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {len(self.collection.get(ids=None))}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store

ChromaDB collection 'pdf_documents' initialized successfully at '../data/vector_store'.
Existing documents in collection: 7


In [119]:
## Generate embeddings for the chunked documents
chunked_texts = [doc.page_content for doc in chunked_docs]
chunked_embeddings = embedding_manager.generate_embeddings(chunked_texts)
vector_store.add_documents(chunked_docs, chunked_embeddings)

Generating embeddings for 72 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  6.41it/s]


Generated embeddings with shape: (72, 384)
Successfully added 72 documents to the vector store.
Total documents in collection after addition: 7


### Retriever Pipeline From VectorStoreDB:

In [120]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 / (1 + distance)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)


In [121]:
rag_retriever.retrieve("What is attention is all you need?")
rag_retriever.retrieve("What is RAG?")

Retrieving documents for query: 'What is attention is all you need?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 94.22it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
Retrieving documents for query: 'What is RAG?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 139.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fd7227b1_0',
  'content': 'RAGTechnique\nFebruary 2024',
  'metadata': {'total_pages': 20,
   'title': 'RAG - Technique',
   'source_file': 'rag-deck.pdf',
   'doc_index': 0,
   'moddate': "D:20240228114547Z00'00'",
   'file_type': 'pdf',
   'source': '../data/pdf/rag-deck.pdf',
   'creationdate': "D:20240228114547Z00'00'",
   'content_length': 26,
   'page_label': '1',
   'producer': 'macOS Version 14.3.1 (Build 23D60) Quartz PDFContext',
   'creator': 'Google',
   'page': 0},
  'similarity_score': 0.4667028481664965,
  'distance': 1.1426910161972046,
  'rank': 1},
 {'id': 'doc_1a3baea2_0',
  'content': 'RAGTechnique\nFebruary 2024',
  'metadata': {'creationdate': "D:20240228114547Z00'00'",
   'title': 'RAG - Technique',
   'page_label': '1',
   'source_file': 'rag-deck.pdf',
   'file_type': 'pdf',
   'source': '../data/pdf/rag-deck.pdf',
   'content_length': 26,
   'creator': 'Google',
   'moddate': "D:20240228114547Z00'00'",
   'total_pages': 20,
   'page': 0,
   'doc_i

### Integration Vectordb Context Pipeline With LLM output

In [ ]:
### Simple pipeline with Groq LLM

from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()

### Intialize the Groq LLM with your API key
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please set it in your .env file.")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.7)

## 2. Simple RAG function: retieve context + generate response
def rag_simple(query, retriever, llm, top_k=5):
    """Simple RAG function that retrieves context and generates a response using LLM.
    
    Args:
        query (str): The input query for retrieval and generation.
        retriever (RAGRetriever): The retriever instance to fetch relevant documents.
        llm (ChatGroq): The language model instance for generating responses.
        top_k (int): Number of top documents to retrieve.
    Returns:
        str: The generated response from the LLM based on retrieved context.
    """
    ## generate the context 
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "" 
    if not context:
        return "No relevant context found for the query."
    
    ## generate the answer using the LLM
    prompt = f""" Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
            
    response = llm.invoke([prompt.format(query=query, context=context)])
    return response.content if response else "No response generated by the LLM."
    

In [ ]:
answer = rag_simple("What is attention is all you need?", rag_retriever, llm)
print(f"Answer: {answer}")

Retrieving documents for query: 'What is attention is all you need?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
Answer: "Attention is All You Need" is a research paper by Vaswani et al. (2017) that introduced the Transformer architecture, a novel approach to neural machine translation that relies entirely on self-attention mechanisms, eliminating the need for recurrent neural networks (RNNs) and convolutional neural networks (CNNs).


### List all available models in the Groq API

In [ ]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096, 'hugging_face_id': 'ALLaM-AI/ALLaM-2.0-7B-Instruct', 'name': 'ALLaM-2-7b', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 4096, 'max_output_length': 4096, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['json_mode']}, {'id': 'llama-3.1-8b-instant', 'object': 'model', 'created': 1693721698, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 131072, 'hugging_face_id': 'meta-llama/Llama-3.1-8B-Instruct', 'name': 'Llama 3.1 8B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 131072, 'pricing': {'prompt': '0.00000005', 'completion': '0.00000008', 'image': '0', 'request': '0', 'inp